# 02A 模型、工具与执行程序
对应 **L02.01–L02.04**。所有文件工具都会实际读取课程资料；这个 Notebook 不调用模型服务。

目标：区分单次调用、固定工作流与动态循环；写清工具协议；检查调用 ID 与真实结果的关系。


## 学习路线与配套课件

实践 2A：工具声明与真实执行（`L02-P01`）。

建议先完成本节概念正课，再进入本实践小节。这个 Notebook 可用独立新内核从头运行。先预测，再执行代码、修改一个条件并解释结果。

对应课件内容：定义与边界、系统组成、Workflow 与 Agent 的区别，以及工具调用协议。

按“问题—代码—观察—练习—可复用结果”的顺序学习。先完成练习，再展开参考分析。回放、保存的真实记录和新的在线请求都会明确标注；在线请求默认关闭。

In [1]:
from pathlib import Path
import sys, json, copy
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "02 从大模型到 Agent 组成与最小循环"
sys.path.insert(0, str(CHAPTER / "代码"))
DATA = CHAPTER / "数据/demo-repo"
print("教学资料:", DATA.relative_to(ROOT))


教学资料: 02 从大模型到 Agent 组成与最小循环\数据\demo-repo


## 任务完成需要外部证据
“解释配置是什么”可以只做概念讲解；“当前项目从哪里加载配置”必须看资料。

下面故意给一段看似流畅、却没有证据的回答。检查回答出现过文件名，是否就等于真的读取过文件？


In [2]:
imagined_answer = "配置加载入口是 loader.py 的 load_config。"
actual_files = sorted(p.name for p in DATA.iterdir() if p.is_file())
print("没有读取内容时的示例回答:",imagined_answer)
print("实际目录:",actual_files)
assert "loader.py" not in actual_files
print("仅凭流畅文本不能验收仓库任务。")


没有读取内容时的示例回答: 配置加载入口是 loader.py 的 load_config。
实际目录: ['README.md', 'config.py', 'main.py', 'settings.json']
仅凭流畅文本不能验收仓库任务。


## 目标、模型、上下文、工具、环境、状态、产物
程序保存目标与消息，向模型提供工具声明；模型提出行动，执行程序检查并调用函数；结果进入消息，帮助模型决定下一步。

用普通字典就能观察这些部分。独立规划器、长期记忆或多智能体都不是这个最小案例的必需项。


In [3]:
state = {"goal":"找出配置加载入口", "messages":[], "tool_calls":0, "artifact":None}
state["messages"].append({"role":"user","content":state["goal"]})
print(json.dumps(state,ensure_ascii=False,indent=2))
print("环境是 data/demo-repo；字典状态保存在 Python 进程里。进程退出后不会自动恢复。")


{
  "goal": "找出配置加载入口",
  "messages": [
    {
      "role": "user",
      "content": "找出配置加载入口"
    }
  ],
  "tool_calls": 0,
  "artifact": null
}
环境是 data/demo-repo；字典状态保存在 Python 进程里。进程退出后不会自动恢复。


## 谁决定下一步
固定工作流预先写出读取顺序。Agent 的下一步由模型结合观察选择。下面用一个可复现的规则模拟动态分支，重点观察“新信息改变下一步”的控制方式。

**先预测：**把 main.py 改名后，固定路径流程会在哪里失效？


In [4]:
fixed_plan = [("read_file", "main.py"), ("read_file", "config.py")]
print("固定流程:", fixed_plan)
observed_files = actual_files
adaptive_choice = "main.py" if "main.py" in observed_files else "README.md"
print("依据当前观察选择下一步:",adaptive_choice)
# 判断组织方式看路径的决定者；有 if 分支不自动意味着它是一个 LLM Agent。


固定流程: [('read_file', 'main.py'), ('read_file', 'config.py')]
依据当前观察选择下一步: main.py


## 工具声明与真正执行
声明描述工具名、使用时机、参数和返回约定。模型看见声明，不会因此获得 Python 执行权限。下面是完整声明与处理函数。

本案例限制为课程自带的四个文本文件，检查相对路径和符号链接。它是教学范围控制，不是生产级沙箱。


In [5]:
def declaration(name, description, properties, required):
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": properties,
                           "required": required, "additionalProperties": False}}}


TOOLS = [
    declaration("list_files", "列出教学仓库可读取的相对路径。未知目录结构时先使用。", {}, []),
    declaration("read_file", "读取一个已列出的文本文件，返回实际行号；不存在时返回错误。",
                {"path": {"type": "string", "description": "list_files 返回的相对文件路径"}}, ["path"]),
    declaration("search_text", "在教学仓库逐行查找字面关键词；大小写敏感，空结果表示未命中。",
                {"query": {"type": "string", "description": "非空字面字符串，不是正则表达式"}}, ["query"]),
]


class RepoTools:
    def __init__(self, root):
        self.root = Path(root).resolve()
        # 只允许课程自带的这几个文件；不遍历用户目录。
        self.allowed = ("README.md", "main.py", "config.py", "settings.json")
        self.registry = {"list_files": self.list_files, "read_file": self.read_file, "search_text": self.search_text}

    def list_files(self):
        return {"files": [p for p in self.allowed if (self.root / p).is_file() and not (self.root / p).is_symlink()]}

    def read_file(self, path):
        candidate = self.root / path
        if path not in self.allowed or candidate.is_symlink() or candidate.resolve().parent != self.root:
            raise PermissionError("只允许读取 list_files 列出的课程文件")
        if candidate.stat().st_size > 16000:
            raise ValueError("文件超过 16000 字节，请缩小教学资料")
        lines = candidate.read_text(encoding="utf-8").splitlines()
        return {"path": path, "lines": [{"line": i+1, "text": t} for i, t in enumerate(lines)]}

    def search_text(self, query):
        hits = []
        for path in self.list_files()["files"]:
            for line in self.read_file(path)["lines"]:
                if query in line["text"]:
                    hits.append({"path": path, **line})
        return {"query": query, "hits": hits[:30], "truncated": len(hits) > 30}

    def execute(self, name, arguments):
        # 这里实现本课实际使用的 schema 子集，不冒充完整 JSON Schema 验证器。
        specs = {t["function"]["name"]: t["function"]["parameters"] for t in TOOLS}
        if name not in self.registry:
            return {"ok": False, "error": {"code": "unknown_tool", "message": "工具未注册"}}
        try:
            args = json.loads(arguments) if isinstance(arguments, str) else arguments
            spec = specs[name]
            if not isinstance(args, dict) or set(args) != set(spec["required"]):
                raise ValueError("参数字段必须与声明一致")
            if any(not isinstance(v, str) or not v.strip() or len(v) > 200 for v in args.values()):
                raise ValueError("本课工具的参数必须是 1–200 字符的非空字符串")
            return {"ok": True, "data": self.registry[name](**args)}
        except (json.JSONDecodeError, ValueError, TypeError):
            return {"ok": False, "error": {"code": "invalid_arguments", "message": "请检查参数字段、类型和格式"}}
        except PermissionError:
            return {"ok": False, "error": {"code": "out_of_scope", "message": "只允许读取课程仓库已列出的文件"}}
        except FileNotFoundError:
            return {"ok": False, "error": {"code": "not_found", "message": "文件不存在，请重新列目录或搜索"}}
        except OSError:
            return {"ok": False, "error": {"code": "io_error", "message": "读取失败，请检查教学文件是否可用"}}


In [6]:
repo_tools = RepoTools(DATA)
print("给模型的 read_file 声明:")
print(json.dumps(TOOLS[1],ensure_ascii=False,indent=2))
print("真实目录:",repo_tools.execute("list_files",{}))
result = repo_tools.execute("read_file",{"path":"main.py"})
for line in result["data"]["lines"]:
    print(f"main.py:{line['line']}: {line['text']}")


给模型的 read_file 声明:
{
  "type": "function",
  "function": {
    "name": "read_file",
    "description": "读取一个已列出的文本文件，返回实际行号；不存在时返回错误。",
    "parameters": {
      "type": "object",
      "properties": {
        "path": {
          "type": "string",
          "description": "list_files 返回的相对文件路径"
        }
      },
      "required": [
        "path"
      ],
      "additionalProperties": false
    }
  }
}
真实目录: {'ok': True, 'data': {'files': ['README.md', 'main.py', 'config.py', 'settings.json']}}
main.py:1: """StudyBox 运行入口。"""
main.py:2: from config import load_settings
main.py:3: 
main.py:4: 
main.py:5: def main():
main.py:6:     settings = load_settings()
main.py:7:     print(f"欢迎，{settings['course']}！")
main.py:8: 
main.py:9: 
main.py:10: if __name__ == "__main__":
main.py:11:     main()


## 调用 ID 关联行动与观察
一次调用的名字并不足以标识结果：同一工具可以读不同文件。使用工具调用 ID 把每个返回值对应回原请求。


In [7]:
call = {"id":"call_001","type":"function","function":{"name":"read_file","arguments":'{"path":"main.py"}'}}
output = repo_tools.execute(call["function"]["name"],call["function"]["arguments"])
tool_message = {"role":"tool","tool_call_id":call["id"],"content":json.dumps(output,ensure_ascii=False)}
assert tool_message["tool_call_id"] == call["id"]
print("关联 ID:",tool_message["tool_call_id"],"真实读取成功:",output["ok"])


关联 ID: call_001 真实读取成功: True


### 两个同名工具调用，如何不串结果

配套课件：L02.04-S03、L02.04-S04。

模型可以两次调用 read_file。函数名相同，参数和调用 ID 不同；回传必须用 tool_call_id 指回原调用。下面真实读取两个课程文件，再故意反转展示顺序，检查关联仍正确。

In [8]:
case_calls = [("read-main", "main.py"), ("read-config", "config.py")]
case_results = []
for case_id, case_path in case_calls:
    case_result = repo_tools.execute("read_file", {"path": case_path})
    assert case_result["ok"]
    case_results.append({"role": "tool", "tool_call_id": case_id,
                         "content": json.dumps(case_result, ensure_ascii=False)})
for case_message in reversed(case_results):
    case_body = json.loads(case_message["content"])
    print(case_message["tool_call_id"], "→", case_body["data"]["path"])
    assert dict(case_calls)[case_message["tool_call_id"]] == case_body["data"]["path"]


read-config → config.py
read-main → main.py


**结果解读**

read-config 对应 config.py，read-main 对应 main.py；展示顺序反转没有改变关联。ID 解决“谁的结果”，不能单独保证正文正确、消息完整或符合服务的消息顺序要求。

**小练习**

把第二个结果的 tool_call_id 改为 read-main，关联检查能否发现问题？在真实循环里还应该检查什么？

<details><summary>完成后展开参考分析</summary>

路径对应断言失败。循环还需拒绝重复 ID、缺失结果和未知 ID；第四节进一步检查工具消息组完整性。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

### 练习：搜索入口函数
调用 search_text 查找 `load_settings`，打印每个命中的路径与行号。再搜索不存在的词，说明空结果与工具出错有什么差别。


In [9]:
# TODO：调用 repo_tools.execute，并遍历 data.hits。
student_result = None
print("你的搜索结果:",student_result)


你的搜索结果: None


In [10]:
for query in ["load_settings","def no_such_function"]:
    found = repo_tools.execute("search_text",{"query":query})
    assert found["ok"]
    print("关键词:",query,"命中:",len(found["data"]["hits"]))
    for hit in found["data"]["hits"]:
        print(f"{hit['path']}:{hit['line']}: {hit['text']}")


关键词: load_settings 命中: 3
main.py:2: from config import load_settings
main.py:6:     settings = load_settings()
config.py:7: def load_settings():
关键词: def no_such_function 命中: 0


## 实用任务：生成可复查的证据包
真实应用通常不会把原始工具返回直接塞进最终答案。先把命中状态、来源位置和原文整理成稳定结构，后续模型、界面或验收程序都可以复用。


In [11]:
def build_evidence_bundle(query):
    result = repo_tools.execute("search_text", {"query": query})
    if not result["ok"]:
        return {"query": query, "status": "error", "citations": [], "error": result["error"]}
    hits = result["data"]["hits"]
    return {
        "query": query,
        "status": "found" if hits else "empty",
        "citations": [
            {"path": hit["path"], "line": hit["line"], "text": hit["text"]}
            for hit in hits
        ],
    }

bundle = build_evidence_bundle("load_settings")
print(json.dumps(bundle, ensure_ascii=False, indent=2))
assert bundle["status"] == "found"
assert all({"path", "line", "text"} <= set(item) for item in bundle["citations"])


{
  "query": "load_settings",
  "status": "found",
  "citations": [
    {
      "path": "main.py",
      "line": 2,
      "text": "from config import load_settings"
    },
    {
      "path": "main.py",
      "line": 6,
      "text": "    settings = load_settings()"
    },
    {
      "path": "config.py",
      "line": 7,
      "text": "def load_settings():"
    }
  ]
}


## 提交与自检
提交工具声明、一次成功调用、一次空结果，解释声明与执行的分工。

思考：模型说“已读取”但没有 tool 消息，应该如何处理？为什么要在程序中验证参数？

接着打开 **02-loop-and-observations.ipynb**。
